#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

#Read Bronze Table

In [0]:
df = spark.table("demo_project.bronze.crm_cust_info")

##Analyzing the Data

In [0]:
df.limit(10).display()
df.count()

#Silver Tranformations

##Renaming Columns

In [0]:
rename_map = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks

In [0]:
df.limit(10).display()

##Remove Records with Customer ID Missing

In [0]:
df = df.filter(col("customer_id").isNotNull())

##Sanity checks

In [0]:
df.count()

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

##Normalization

In [0]:
df = (
    df.withColumn(
        "marital_status",
        F.when(F.upper(col("marital_status")).contains("M"), "Married")
        .when(F.upper(col("marital_status")).contains("S"), "Single")
        .otherwise(col("marital_status")),
    )
    .withColumn(
        "gender",
        F.when(F.upper(col("gender")).contains("M"), "Male")
        .when(F.upper(col("gender")).contains("F"), "Female")
        .otherwise("N/A")
    )
)

##Sanity checks


In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("demo_project.silver.crm_customers")

##Sanity checks

In [0]:
%sql
select * from demo_project.silver.crm_customers limit 10;